In [2]:
print("hello")

hello


In [ ]:
from embedder import Embedder

embed = Embedder()

#################
##  Q1 Embed the query What's the first value (v[0])?
#################
q1 = "How does approximate nearest neighbor search work?"
v1 = embed.encode(q1)
print(len(v1))
print(v1[0])

384
-0.02058203437252893


In [6]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [10]:
content = ''
for doc in documents:
    if doc['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md' :
        content = doc['content']

In [11]:
vContent = embed.encode(content)

In [14]:
#####################
### Q2 Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?
####################
v1.dot(vContent)

np.float64(0.36107027225589694)

In [23]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [20]:
batch_vectors = []
chks_content = []
for chk in chunks:
    chks_content.append(chk['content'])
batch_vectors = embed.encode_batch(chks_content)


len(batch_vectors)

295

In [26]:
import numpy as np
X = np.array(batch_vectors)
print(X)

[[-0.08756474  0.0183638  -0.08122425 ...  0.0305382  -0.02172767
   0.03277498]
 [ 0.02436192 -0.10619479  0.03307313 ...  0.01430086 -0.0012554
   0.04325694]
 [-0.01780481  0.03103092  0.00856104 ...  0.02220219 -0.03375532
   0.04288225]
 ...
 [ 0.00980345  0.04912257  0.01207493 ... -0.09453998 -0.06321277
   0.04775796]
 [-0.03622024  0.06821856 -0.01540895 ... -0.00271633  0.01875558
   0.01007469]
 [-0.02975661 -0.00552575 -0.03531848 ...  0.01044234  0.02297965
  -0.01966068]]


In [22]:
X.shape

(295, 384)

In [24]:
scores = X.dot(v1)

In [27]:
idx = np.argmax(scores)

In [30]:
idx, scores[idx]

(np.int64(94), np.float64(0.6489017718578813))

In [31]:
###########################
##  Q3 Which file does the highest-scoring chunk belong to (its filename)?
############################
chunks[94]

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 